# Ejercicios: verificación de certificados en NP

Este cuaderno ilustra la diferencia entre encontrar una solución y verificar una solución.

En los problemas de NP, si alguien entrega un certificado candidato, podemos comprobarlo eficientemente.

## Ejercicio 1: SUBSET-SUM

El certificado es una lista de índices. Verificarlo consiste en sumar los números elegidos y comparar con el objetivo.

In [ ]:
def verificar_subset_sum(numeros, objetivo, certificado):
    if any(indice < 0 or indice >= len(numeros) for indice in certificado):
        return False
    if len(set(certificado)) != len(certificado):
        return False
    suma = sum(numeros[indice] for indice in certificado)
    return suma == objetivo


numeros = [3, 8, 12, 4, 7, 2]
objetivo = 15

for certificado in [[1, 4], [0, 3, 5], [2, 5], [0, 0, 2]]:
    print(certificado, verificar_subset_sum(numeros, objetivo, certificado))

Prueba nuevos certificados. ¿Puedes encontrar otro subconjunto que sume `15`?

## Ejercicio 2: satisfacibilidad booleana

Representaremos una fórmula en CNF como lista de cláusulas. Cada literal es una cadena: `"x"` significa variable positiva y `"¬x"` significa variable negada.

In [ ]:
def evaluar_literal(literal, asignacion):
    if literal.startswith("¬"):
        return not asignacion[literal[1:]]
    return asignacion[literal]


def verificar_cnf(formula, asignacion):
    for clausula in formula:
        if not any(evaluar_literal(literal, asignacion) for literal in clausula):
            return False
    return True


formula = [
    ["x", "y", "¬z"],
    ["¬x", "z"],
    ["¬y", "z"],
]

asignaciones = [
    {"x": True, "y": True, "z": True},
    {"x": True, "y": False, "z": False},
    {"x": False, "y": False, "z": True},
]

for asignacion in asignaciones:
    print(asignacion, verificar_cnf(formula, asignacion))

Modifica una cláusula o una asignación. Observa que verificar una asignación es directo aunque encontrar una buena pueda requerir búsqueda.

## Ejercicio 3: CLIQUE

El certificado es un conjunto de vértices. Verificarlo consiste en comprobar que todos los pares están conectados.

In [ ]:
def arista(grafo, u, v):
    return v in grafo.get(u, set()) and u in grafo.get(v, set())


def verificar_clique(grafo, certificado, k):
    if len(certificado) != k:
        return False
    vertices = list(certificado)
    if len(set(vertices)) != len(vertices):
        return False
    for i, u in enumerate(vertices):
        for v in vertices[i + 1:]:
            if not arista(grafo, u, v):
                return False
    return True


grafo = {
    "A": {"B", "C", "D"},
    "B": {"A", "C", "D"},
    "C": {"A", "B"},
    "D": {"A", "B", "E"},
    "E": {"D"},
}

for certificado in [{"A", "B", "C"}, {"A", "B", "D"}, {"A", "D", "E"}]:
    print(certificado, verificar_clique(grafo, certificado, k=3))

## Preguntas de cierre

1. ¿Qué parte de cada problema actúa como certificado?
2. ¿Por qué verificar no es lo mismo que encontrar?
3. ¿Cómo crecería una búsqueda exhaustiva para SAT si añadimos más variables?

In [ ]:
# ── Verificación automática ──────────────────────────────────────────────────

def verificar_3sat(formula, asig):
    """Verifica si una asignación satisface una fórmula 3-SAT."""
    return all(any((l > 0 and asig.get(abs(l), False)) or
                   (l < 0 and not asig.get(abs(l), True)) for l in c) for c in formula)

# Caso satisfacible
f = [[1, -2, 3], [-1, 2, -3]]
assert verificar_3sat(f, {1: True, 2: True, 3: False}), "Asignacion valida rechazada"
# Caso no satisfacible
assert not verificar_3sat([[1], [-1]], {1: True}), "Contradiccion aceptada"
print("✓ Verificador de 3-SAT correcto")